# 🏬 Competitive-Intelligence Agent: A Structured Market Brief with Sources

## Learning Objectives
In this notebook, you will learn:
1. **A domain persona for a research agent** - A system prompt that turns a generic web-search agent into a retail competitive-intelligence analyst
2. **Structured output from an agent** - `create_agent(response_format=...)` returns a typed brief (competitors, opportunities, risks, data gaps) instead of free text
3. **Honest data gaps** - Real-time footfall is rarely on the web; the brief must say so instead of inventing numbers
4. **Grounding check in code** - Verify that every source URL in the brief came from this run's own search results

## Where this fits
`01_research_assistant_agent.ipynb` built a general research assistant. This notebook specializes the same idea for one business use: monitoring clothing-store competitors in busy urban areas for owners, marketers, location analysts and investors. Structured output with `create_agent` was introduced in `04_langchain_agents_and_middleware/03_structured_output.ipynb`; here it makes the agent's answer checkable.

## Prerequisites
- `create_agent` and structured output from Modules 03–04 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the competitive-intelligence analyst
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🎭 Part 1: The Analyst Persona and Its Search Tool

The agent has one tool, Tavily web search. What makes it a competitive-intelligence analyst is the system prompt: who it serves, what insights it produces, and how it presents them.

The original prompt for this assistant ended with *"If real-time data is requested, explain how it would be obtained and provide example insights."* "Example insights" invites the model to make up footfall figures and peak hours that look like data. The prompt below keeps the persona and replaces that line with an explicit rule: **no data → report a data gap**.

In [ ]:
# ============================================================================
# PERSONA PROMPT + SEARCH TOOL
# ============================================================================
from langchain_tavily import TavilySearch

PERSONA_PROMPT = """You are a Conversational AI Assistant designed to help businesses, marketers, analysts, and investors monitor and analyze competitor activity in retail, with a special focus on clothing stores in high-traffic urban locations.

Your core objectives are:

Competitor Insights – Provide real-time information on nearby clothing store competitors, including store footfall, busiest times, and peak customer hours.

Business Strategy Support – Generate actionable insights that help business owners and managers adjust strategies to improve market positioning.

Marketing Optimization – Assist marketing teams in identifying trends, optimizing promotions, and aligning campaigns with or against competitor activities.

Location Analysis – Provide data to real estate and location analysts on high-footfall times and ideal store placement for new launches.

Investment Analysis – Support investors and market analysts in assessing business potential, risks, and opportunities in competitive retail areas.

Guidelines for Response:

Always provide insights in a clear, structured, and business-oriented format.

Where possible, highlight trends, risks, and opportunities.

Adapt explanations for the intended beneficiary (business owner, marketer, analyst, or investor).

Use a professional, analytical, and action-driven tone.

Base every claim on your web searches. If real-time or location-specific data (footfall, busiest hours) is not in the search results, do not estimate or invent it: list it under data gaps and say how it could be obtained (e.g., footfall analytics, Google Maps popular times, store visits)."""

search_tool = TavilySearch(max_results=5)
tools = [search_tool]

print(f"✅ Persona prompt: {len(PERSONA_PROMPT.split())} words")
print(f"✅ Tools: {[t.name for t in tools]}")

---
## 📋 Part 2: The Brief as Structured Output

A free-text answer is hard to act on and impossible to check. The brief below has a fixed shape: each competitor comes with the URL that supports it, and anything the searches could not establish goes into `data_gaps`.

### Key Concepts:
- **`response_format`**: `create_agent` runs the normal tool loop, then returns the final answer as an instance of the schema in `result["structured_response"]`
- **Evidence per claim**: asking for `evidence_url` on every competitor makes each claim traceable to a source

In [ ]:
# ============================================================================
# BRIEF SCHEMA + AGENT
# ============================================================================
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class Competitor(BaseModel):
    name: str = Field(description="Store or brand name")
    positioning: str = Field(description="Price segment, style and target customer, from the sources")
    evidence_url: str = Field(description="URL of the search result that supports this entry")


class CompetitorBrief(BaseModel):
    area: str = Field(description="The area the brief covers")
    audience: str = Field(description="Who the brief is written for: business owner, marketer, analyst or investor")
    competitors: list[Competitor] = Field(description="Competitors found in the searches (max 5)")
    opportunities: list[str] = Field(description="Actionable opportunities (max 3)")
    risks: list[str] = Field(description="Risks to watch (max 3)")
    data_gaps: list[str] = Field(description="Requested information the searches did not provide")


ci_agent = create_agent(llm, tools, system_prompt=PERSONA_PROMPT, response_format=CompetitorBrief)

print(f"✅ Agent graph nodes: {list(ci_agent.get_graph().nodes)}")
print(f"✅ Brief fields: {list(CompetitorBrief.model_fields)}")

---
## 🔎 Part 3: Running the Analyst

Two requests from the original use cases: a competitor scan that asks for busiest hours (data that is rarely published), and a location question for a store launch. The helper prints the brief and returns the full result, so the next part can inspect the searches behind it.

In [ ]:
# ============================================================================
# RUN HELPER: invoke the agent and print the brief
# ============================================================================
def run_brief(question: str):
    result = ci_agent.invoke({"messages": [{"role": "user", "content": question}]})
    brief: CompetitorBrief = result["structured_response"]
    searches = [tc["args"]["query"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])
                if tc["name"] == search_tool.name]
    print(f"❓ {question}\n🔎 searches: {searches}\n")
    print(f"📍 {brief.area}  (for: {brief.audience})")
    for c in brief.competitors:
        print(f"  • {c.name}: {c.positioning}\n      source: {c.evidence_url}")
    print("💡 Opportunities:", *brief.opportunities, sep="\n  - ")
    print("⚠️ Risks:", *brief.risks, sep="\n  - ")
    print("🕳️ Data gaps:", *(brief.data_gaps or ["(none reported)"]), sep="\n  - ")
    return result


result_1 = run_brief("Who are the top 5 clothing store competitors near Koramangala, Bangalore, "
                     "and when are their busiest hours?")

In [ ]:
# ============================================================================
# SECOND REQUEST: location analysis for a new store
# ============================================================================
result_2 = run_brief("Identify the best locations in Whitefield, Bangalore for launching a new "
                     "mid-range clothing store, based on competitor activity.")

Look at the **data gaps** of the first brief: store-level busiest hours are almost never published, so an honest brief lists them as missing (with a way to collect them) rather than filling the gap with plausible numbers.

---
## ✅ Part 4: Checking the Sources

A brief is only as good as its sources. The check below collects every URL that the search tool actually returned during the run (from the `ToolMessage`s in the agent's state) and tests each `evidence_url` against that set. A URL that is not in the set was not seen by the agent in this run — it came from the model's memory or was invented.

In [ ]:
# ============================================================================
# GROUNDING CHECK: every evidence_url must come from this run's search results
# ============================================================================
import json

from langchain_core.messages import ToolMessage


def search_urls(result) -> set[str]:
    """All URLs the search tool returned during one agent run."""
    urls = set()
    for m in result["messages"]:
        if isinstance(m, ToolMessage) and m.name == search_tool.name:
            try:
                urls |= {hit["url"] for hit in json.loads(m.content)["results"]}
            except (json.JSONDecodeError, KeyError, TypeError):
                pass
    return urls


def check_sources(label: str, brief: CompetitorBrief, result) -> None:
    seen = search_urls(result)
    grounded = [c for c in brief.competitors if c.evidence_url in seen]
    print(f"{label}: {len(seen)} URLs returned by search, "
          f"{len(grounded)}/{len(brief.competitors)} competitors cite one of them")
    for c in brief.competitors:
        if c.evidence_url not in seen:
            print(f"   ❌ not from this run's searches: {c.name} → {c.evidence_url}")


for label, result in [("Koramangala", result_1), ("Whitefield", result_2)]:
    check_sources(label, result["structured_response"], result)

To see the check fail, add one entry the searches never returned — the kind of confident, plausible line a model can produce from memory — and run the same function on it.

In [ ]:
# ============================================================================
# THE CHECK CATCHES AN UNSUPPORTED ENTRY
# ============================================================================
doctored = result_1["structured_response"].model_copy(deep=True)
doctored.competitors.append(Competitor(
    name="Urban Threads Koramangala",
    positioning="Mid-range streetwear, busiest 6-9 pm on weekends",
    evidence_url="https://example.com/koramangala-footfall-report",
))
check_sources("Koramangala + invented entry", doctored, result_1)

Any ❌ line is a claim to verify by hand before it goes into a report. The check is cheap and deterministic, so it can run on every brief the agent produces.

---
## 📝 Summary

In this notebook, we learned:

### 1. A domain persona
- The system prompt, not the tools, turns a web-search agent into a competitive-intelligence analyst
- A prompt that asks for "example insights" when data is missing asks the model to invent data — replace it with an explicit data-gap rule

### 2. Structured output from an agent
- `create_agent(response_format=Schema)` keeps the tool loop and returns a typed brief in `result["structured_response"]`

### 3. Honest data gaps
- Missing information is a field of the output, so "no data" cannot be mistaken for "nothing to report"

### 4. Grounding check
- Compare every cited URL with the URLs the tool really returned in the run; anything else is flagged for review

### Next Steps
- **`03_financial_analyst_agent.ipynb`**: an agent over structured market-data tools (OpenBB) that handles follow-up questions and declines out-of-scope requests